# DCE-MRI — ROI quality control

Independent verification of DCE-MRI crop generation, masks, patient identifiers, dataset partitions, and ROI integrity.


In [ ]:


from pathlib import Path

MAMA_MIA_ROOT = Path("/kaggle/input/mama-mia")

OUT_ROOT = Path("/kaggle/working/ROI_MRI_Crops_256_v1")
NPZ_DIR = OUT_ROOT / "npz"
LOG_DIR = OUT_ROOT / "logs"
FIG_DIR = OUT_ROOT / "figures"
AUDIT_DIR = OUT_ROOT / "audit"

RUN_PRECHECK = True
RUN_GENERATION = False
RUN_AUDIT = True
CREATE_ZIP = True

IMAGE_SIZE = 256
ROI_MARGIN = 0.40
SPLIT_SEED = 42

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

EXTERNAL_DATASET_NAMES = {"duke"}

DEV_DATASET_NAMES = {"ispy1", "ispy2", "nact"}

P_LOW = 1.0
P_HIGH = 99.0
NORMALIZE_USING_ALL_PHASES = True

EARLY_POST_INDEX = 1

PHASE1_NO_TRAINING = True
assert PHASE1_NO_TRAINING is True

for d in [OUT_ROOT, NPZ_DIR, LOG_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUT_ROOT:", OUT_ROOT)
print("RUN_GENERATION:", RUN_GENERATION)


## Étape 1 — Imports et dépendances


In [ ]:


import os
import re
import json
import math
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import nibabel as nib
except ImportError as e:
    raise ImportError(
        "nibabel n'est pas installé. Dans Kaggle, ajoute une cellule: !pip install nibabel -q "
        "ou utilise une image Kaggle qui contient nibabel."
    ) from e

try:
    import cv2
except ImportError as e:
    raise ImportError("opencv-python / cv2 est requis pour resize et morphologie.") from e

try:
    from sklearn.model_selection import train_test_split
except ImportError as e:
    raise ImportError("scikit-learn est requis pour le split patient-level.") from e

from scipy import ndimage as ndi

print("Imports OK")


## Étape 2 — Détection du dossier MAMA-MIA


In [ ]:


def find_existing_root(candidate: Path) -> Path:
    if candidate.exists():
        return candidate
    roots = list(Path("/kaggle/input").glob("*"))
    roots = [p for p in roots if p.is_dir()]
    if not roots:
        raise FileNotFoundError("Aucun input Kaggle trouvé dans /kaggle/input.")
    print("MAMA_MIA_ROOT non trouvé. Inputs disponibles:")
    for p in roots:
        print(" -", p)
    counts = []
    for p in roots:
        n = len(list(p.rglob("*.nii"))) + len(list(p.rglob("*.nii.gz")))
        counts.append((n, p))
    counts.sort(reverse=True, key=lambda x: x[0])
    if counts[0][0] == 0:
        raise FileNotFoundError("Aucun fichier NIfTI détecté dans /kaggle/input.")
    print("Racine choisie automatiquement:", counts[0][1], "NIfTI:", counts[0][0])
    return counts[0][1]

MAMA_MIA_ROOT = find_existing_root(MAMA_MIA_ROOT)

nifti_files = sorted(list(MAMA_MIA_ROOT.rglob("*.nii")) + list(MAMA_MIA_ROOT.rglob("*.nii.gz")))
print("MAMA_MIA_ROOT:", MAMA_MIA_ROOT)
print("Nombre total de NIfTI:", len(nifti_files))

pd.DataFrame({"path": [str(p) for p in nifti_files[:30]]}).head(30)


## Étape 3 — Classification image / masque et appariement


In [ ]:


MASK_PATTERNS = [
    "seg", "segmentation", "mask", "label", "annotation", "roi", "lesion"
]

def clean_nii_stem(path: Path) -> str:
    name = path.name
    if name.endswith(".nii.gz"):
        name = name[:-7]
    elif name.endswith(".nii"):
        name = name[:-4]
    return name

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def is_mask_file(path: Path) -> bool:
    text = normalize_text("/".join(path.parts[-5:]))
    return any(pat in text for pat in MASK_PATTERNS)

def detect_dataset_from_path(path: Path) -> str:
    text = normalize_text(str(path))
    if "ispy1" in text or "i_spy1" in text or "ispy_1" in text:
        return "ISPY1"
    if "ispy2" in text or "i_spy2" in text or "ispy_2" in text:
        return "ISPY2"
    if "nact" in text:
        return "NACT"
    if "duke" in text:
        return "Duke"
    return "UNKNOWN"

def remove_mask_tokens(stem: str) -> str:
    s = normalize_text(stem)
    for tok in MASK_PATTERNS:
        s = re.sub(rf"(^|_){tok}($|_)", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def make_pair_key(path: Path) -> str:
    # Clé principale : dataset + dossier parent proche + stem nettoyé.
    dataset = detect_dataset_from_path(path)
    stem = remove_mask_tokens(clean_nii_stem(path))
    parent = normalize_text(path.parent.name)
    grandparent = normalize_text(path.parent.parent.name) if path.parent.parent else ""
    key = f"{dataset}__{grandparent}__{parent}__{stem}"
    key = re.sub(r"(_?phase_?\d+|_?time_?\d+|_?tp_?\d+|_?t\d+$)", "", key)
    key = re.sub(r"_+", "_", key).strip("_")
    return key

image_files = [p for p in nifti_files if not is_mask_file(p)]
mask_files = [p for p in nifti_files if is_mask_file(p)]

print("Images candidates:", len(image_files))
print("Masks candidates:", len(mask_files))

img_df = pd.DataFrame({
    "path": [str(p) for p in image_files],
    "dataset": [detect_dataset_from_path(p) for p in image_files],
    "key": [make_pair_key(p) for p in image_files],
})
msk_df = pd.DataFrame({
    "path": [str(p) for p in mask_files],
    "dataset": [detect_dataset_from_path(p) for p in mask_files],
    "key": [make_pair_key(p) for p in mask_files],
})

pairs = []
for _, row in img_df.iterrows():
    candidates = msk_df[(msk_df["dataset"] == row["dataset"]) & (msk_df["key"] == row["key"])]
    if len(candidates) == 0:
        continue
    mask_path = candidates.iloc[0]["path"]
    pairs.append({
        "dataset": row["dataset"],
        "image_path": row["path"],
        "mask_path": mask_path,
        "pair_key": row["key"],
        "pairing_mode": "exact_key",
    })

pairs_df = pd.DataFrame(pairs)

if len(pairs_df) < max(1, min(len(image_files), len(mask_files)) * 0.25):
    print("Exact key pairing faible. Fallback par proximité de dossier...")
    pairs = []
    for img in image_files:
        ds = detect_dataset_from_path(img)
        local_masks = [m for m in mask_files if detect_dataset_from_path(m) == ds]
        same_parent = [m for m in local_masks if m.parent == img.parent]
        same_grandparent = [m for m in local_masks if m.parent.parent == img.parent.parent]
        cand = same_parent or same_grandparent
        if cand:
            pairs.append({
                "dataset": ds,
                "image_path": str(img),
                "mask_path": str(cand[0]),
                "pair_key": make_pair_key(img),
                "pairing_mode": "folder_fallback",
            })
    pairs_df = pd.DataFrame(pairs)

pairing_report_path = LOG_DIR / "pairing_report.csv"
pairs_df.to_csv(pairing_report_path, index=False)

print("Paires image/masque trouvées:", len(pairs_df))
print("Saved:", pairing_report_path)

display(pairs_df.head(20))

if len(pairs_df) == 0:
    raise RuntimeError(
        "Aucune paire image/masque trouvée. Ajuster les règles d'appariement avant de continuer."
    )


## Étape 4 — Correction / re-dérivation des patient_id


In [ ]:


def derive_volume_id(path: str, dataset: str) -> str:
    p = Path(path)
    text_parts = [normalize_text(x) for x in p.parts]
    stem = normalize_text(clean_nii_stem(p))
    ds = dataset.lower()

    joined = "__".join(text_parts[-8:] + [stem])
    patterns = [
        rf"({ds}[_-]?\d+)",
        r"(ispy1[_-]?\d+)",
        r"(ispy2[_-]?\d+)",
        r"(nact[_-]?\d+)",
        r"(duke[_-]?\d+)",
        r"(breast[_-]?mri[_-]?\d+)",
        r"(patient[_-]?\d+)",
        r"(case[_-]?\d+)",
        r"(sub[_-]?\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, joined, flags=re.IGNORECASE)
        if m:
            return normalize_text(m.group(1)).upper()

    for part in reversed(text_parts[-8:]):
        if any(ch.isdigit() for ch in part) and len(part) >= 3:
            return f"{dataset.upper()}_{part.upper()}"

    return f"{dataset.upper()}_{stem.upper()}"

def derive_patient_id(path: str, dataset: str) -> str:
    vol = derive_volume_id(path, dataset)
    v = normalize_text(vol).upper()
    v = re.sub(r"(_?DCE|_?MRI|_?IMAGE|_?IMG|_?SCAN|_?T\d+|_?PHASE_?\d+|_?TP_?\d+)$", "", v)
    v = re.sub(r"_+", "_", v).strip("_")
    if not v.startswith(dataset.upper()):
        v = f"{dataset.upper()}_{v}"
    return v

pairs_df["dataset_norm"] = pairs_df["dataset"].astype(str).str.upper()
pairs_df["volume_id"] = [
    derive_volume_id(p, ds) for p, ds in zip(pairs_df["image_path"], pairs_df["dataset"])
]
pairs_df["patient_id"] = [
    derive_patient_id(p, ds) for p, ds in zip(pairs_df["image_path"], pairs_df["dataset"])
]

pairs_df["old_patient_id_proxy"] = pairs_df["dataset_norm"]

id_counts = (
    pairs_df.groupby("dataset")["patient_id"]
    .nunique()
    .reset_index(name="n_unique_patient_id")
    .sort_values("dataset")
)

display(id_counts)

fix_report = pairs_df[[
    "dataset", "old_patient_id_proxy", "patient_id", "volume_id",
    "image_path", "mask_path"
]].copy()

fix_report_path = LOG_DIR / "ispy1_patient_id_fix_report.csv"
fix_report.to_csv(fix_report_path, index=False)
print("Saved:", fix_report_path)

ispy1_rows = id_counts[id_counts["dataset"].str.upper() == "ISPY1"]
ispy1_n = int(ispy1_rows["n_unique_patient_id"].iloc[0]) if len(ispy1_rows) else 0
print("ISPY1 unique patient_id:", ispy1_n)

if ispy1_n <= 3:
    raise RuntimeError(
        "BLOCAGE: ISPY1 a encore <=3 patient_id uniques. "
        "Le patient_id n'est pas corrigé de manière plausible. "
        "Inspecter ispy1_patient_id_fix_report.csv et ajuster derive_patient_id()."
    )

for ds in ["ISPY2", "NACT", "Duke"]:
    rows = id_counts[id_counts["dataset"].str.upper() == ds.upper()]
    if len(rows):
        print(ds, "unique patient_id:", int(rows.iloc[0]["n_unique_patient_id"]))
    else:
        print(ds, "non détecté dans les paires.")

pairs_df.to_csv(LOG_DIR / "paired_volumes_with_patient_ids.csv", index=False)
print("Patient ID correction/precheck OK")


## Étape 5 — Split patient-level


In [ ]:


def dataset_role(dataset: str) -> str:
    ds = dataset.lower()
    if ds in EXTERNAL_DATASET_NAMES:
        return "external"
    if ds in DEV_DATASET_NAMES:
        return "dev"
    return "unknown"

pairs_df["dataset_lower"] = pairs_df["dataset"].str.lower()
pairs_df["role"] = pairs_df["dataset_lower"].map(dataset_role)

unknown = pairs_df[pairs_df["role"] == "unknown"]["dataset"].unique().tolist()
if unknown:
    raise RuntimeError(f"Datasets inconnus détectés: {unknown}. Définir leur rôle avant de continuer.")

dev_patients = sorted(pairs_df.loc[pairs_df["role"] == "dev", "patient_id"].unique())
external_patients = sorted(pairs_df.loc[pairs_df["role"] == "external", "patient_id"].unique())

if len(dev_patients) == 0:
    raise RuntimeError("Aucun patient de développement détecté.")
if len(external_patients) == 0:
    warnings.warn("Aucun patient Duke/external détecté. Vérifier la présence du dataset externe.")

train_patients, temp_patients = train_test_split(
    dev_patients,
    train_size=TRAIN_FRAC,
    random_state=SPLIT_SEED,
    shuffle=True,
)

val_patients, test_patients = train_test_split(
    temp_patients,
    train_size=VAL_FRAC / (VAL_FRAC + TEST_FRAC),
    random_state=SPLIT_SEED,
    shuffle=True,
)

split_map = {}
for p in train_patients:
    split_map[p] = "train"
for p in val_patients:
    split_map[p] = "validation"
for p in test_patients:
    split_map[p] = "test"
for p in external_patients:
    split_map[p] = "external"

pairs_df["split"] = pairs_df["patient_id"].map(split_map)

if pairs_df["split"].isna().any():
    missing = pairs_df[pairs_df["split"].isna()][["dataset", "patient_id"]].drop_duplicates()
    raise RuntimeError(f"Certains patients n'ont pas de split:\n{missing}")

sets = {
    "train": set(train_patients),
    "validation": set(val_patients),
    "test": set(test_patients),
    "external": set(external_patients),
}

leaks = {
    "train_vs_validation": len(sets["train"] & sets["validation"]),
    "train_vs_test": len(sets["train"] & sets["test"]),
    "validation_vs_test": len(sets["validation"] & sets["test"]),
    "dev_vs_external": len((sets["train"] | sets["validation"] | sets["test"]) & sets["external"]),
}

print("Leakage:", leaks)
if any(v > 0 for v in leaks.values()):
    raise RuntimeError(f"Fuite patient détectée: {leaks}")

split_report = (
    pairs_df.groupby(["dataset", "split"])
    .agg(n_volumes=("volume_id", "nunique"), n_patients=("patient_id", "nunique"))
    .reset_index()
)
display(split_report)

pairs_df.to_csv(LOG_DIR / "paired_volumes_with_splits.csv", index=False)
with open(LOG_DIR / "split_leakage_audit.json", "w", encoding="utf-8") as f:
    json.dump({"leaks": leaks, "split_seed": SPLIT_SEED}, f, indent=2)

print("Split patient-level OK")


## Étape 6 — Fonctions IRM DCE, normalisation commune, crop ROI


In [ ]:


@dataclass
class VolumeLoadResult:
    image_4d: np.ndarray
    mask_3d: np.ndarray
    spacing_mm: Tuple[float, float, float]
    n_phases: int
    phase_indices: Tuple[int, int, int]
    phase_fallback: str
    affine_shape_ok: bool

def load_nifti_array(path: str):
    nii = nib.load(path)
    arr = np.asarray(nii.get_fdata(dtype=np.float32))
    zooms = nii.header.get_zooms()
    spacing = tuple(float(z) for z in zooms[:3])
    return arr, spacing, nii

def align_image_mask_to_xyzt(image: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, bool]:
    affine_shape_ok = True

    if mask.ndim == 4:
        mask = (mask > 0).any(axis=-1).astype(np.uint8)
    elif mask.ndim != 3:
        raise ValueError(f"Mask ndim non supporté: {mask.ndim}, shape={mask.shape}")

    if image.ndim == 3:
        img4 = image[..., None]
    elif image.ndim == 4:
        candidates = []
        for ax in range(4):
            spatial_shape = tuple([image.shape[i] for i in range(4) if i != ax])
            if spatial_shape == tuple(mask.shape):
                candidates.append(ax)
        if len(candidates) == 1:
            t_axis = candidates[0]
            img4 = np.moveaxis(image, t_axis, -1)
        elif image.shape[:3] == mask.shape:
            img4 = image
        else:
            img4 = image
            if image.shape[:3] != mask.shape:
                affine_shape_ok = False
    else:
        raise ValueError(f"Image ndim non supporté: {image.ndim}, shape={image.shape}")

    if img4.shape[:3] != mask.shape:
        raise ValueError(f"Shape image/mask incompatible après alignement: image={img4.shape}, mask={mask.shape}")

    return img4.astype(np.float32), (mask > 0).astype(np.uint8), affine_shape_ok

def select_dce_phases(n_phases: int) -> Tuple[Tuple[int, int, int], str]:
    if n_phases >= 3:
        pre = 0
        early = min(EARLY_POST_INDEX, n_phases - 1)
        late = n_phases - 1
        return (pre, early, late), "none"
    if n_phases == 2:
        return (0, 1, 1), "duplicated_late_from_early_two_phases"
    if n_phases == 1:
        return (0, 0, 0), "duplicated_single_phase_all_channels"
    raise ValueError("n_phases < 1")

def normalize_volume_common_scale(img4: np.ndarray, phase_indices: Tuple[int, int, int]) -> np.ndarray:
    if NORMALIZE_USING_ALL_PHASES:
        base = img4
    else:
        base = img4[..., list(phase_indices)]

    vals = base[np.isfinite(base)]
    if vals.size == 0:
        raise ValueError("Volume sans valeurs finies.")

    lo, hi = np.percentile(vals, [P_LOW, P_HIGH])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(vals))
        hi = float(np.nanmax(vals))
    if hi <= lo:
        return np.zeros_like(img4[..., list(phase_indices)], dtype=np.float32)

    selected = img4[..., list(phase_indices)].astype(np.float32)
    selected = np.clip(selected, lo, hi)
    selected = (selected - lo) / (hi - lo)
    return np.clip(selected, 0, 1).astype(np.float32)

def load_volume_record(row: pd.Series) -> VolumeLoadResult:
    image, spacing, img_nii = load_nifti_array(row["image_path"])
    mask, mask_spacing, mask_nii = load_nifti_array(row["mask_path"])

    img4, mask3, shape_ok = align_image_mask_to_xyzt(image, mask)
    n_phases = int(img4.shape[-1])
    phase_indices, fallback = select_dce_phases(n_phases)
    img3ch = normalize_volume_common_scale(img4, phase_indices)

    return VolumeLoadResult(
        image_4d=img3ch,
        mask_3d=mask3,
        spacing_mm=spacing,
        n_phases=n_phases,
        phase_indices=phase_indices,
        phase_fallback=fallback,
        affine_shape_ok=shape_ok,
    )

def bbox_from_mask(mask2d: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1

def square_crop_coords(x0, y0, x1, y1, H, W, margin=0.4):
    bw = x1 - x0
    bh = y1 - y0
    side = max(bw, bh)
    side = int(math.ceil(side * (1 + 2 * margin)))
    side = max(side, 1)

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    nx0 = int(math.floor(cx - side / 2))
    ny0 = int(math.floor(cy - side / 2))
    nx1 = nx0 + side
    ny1 = ny0 + side

    touches = nx0 < 0 or ny0 < 0 or nx1 > W or ny1 > H
    return nx0, ny0, nx1, ny1, side, touches

def crop_pad_resize_image_mask(img2d3: np.ndarray, mask2d: np.ndarray, coords):
    H, W = mask2d.shape
    x0, y0, x1, y1, side, touches = coords

    pad_left = max(0, -x0)
    pad_top = max(0, -y0)
    pad_right = max(0, x1 - W)
    pad_bottom = max(0, y1 - H)

    if any(v > 0 for v in [pad_left, pad_top, pad_right, pad_bottom]):
        img_pad = np.pad(
            img2d3,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="constant",
            constant_values=0,
        )
        mask_pad = np.pad(
            mask2d,
            ((pad_top, pad_bottom), (pad_left, pad_right)),
            mode="constant",
            constant_values=0,
        )
        x0p, x1p = x0 + pad_left, x1 + pad_left
        y0p, y1p = y0 + pad_top, y1 + pad_top
    else:
        img_pad = img2d3
        mask_pad = mask2d
        x0p, x1p, y0p, y1p = x0, x1, y0, y1

    img_crop = img_pad[y0p:y1p, x0p:x1p, :]
    mask_crop = mask_pad[y0p:y1p, x0p:x1p]

    img_res = cv2.resize(img_crop, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    mask_res = cv2.resize(mask_crop.astype(np.uint8), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)

    img_res = np.clip(img_res, 0, 1).astype(np.float32)
    mask_res = (mask_res > 0).astype(np.uint8)

    return img_res, mask_res

def lesion_ratio(mask: np.ndarray) -> float:
    return float((mask > 0).mean())


## Étape 7 — Pré-check visuel d’un volume


In [ ]:


def overlay_mask_gray(img, mask, alpha=0.45):
    g = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    rgb = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
    over = rgb.copy()
    over[mask > 0] = [255, 0, 0]
    return cv2.addWeighted(over, alpha, rgb, 1-alpha, 0)

if RUN_PRECHECK:
    sample_row = pairs_df.iloc[0]
    print("Sample:", sample_row[["dataset", "patient_id", "volume_id", "split"]].to_dict())
    vol = load_volume_record(sample_row)

    z_indices = np.where(vol.mask_3d.reshape(-1, vol.mask_3d.shape[-1]).sum(axis=0) > 0)[0]
    if len(z_indices) == 0:
        raise RuntimeError("Le masque du sample pré-check ne contient aucune coupe lésionnée.")
    z = int(z_indices[len(z_indices)//2])

    img_slice = vol.image_4d[:, :, z, :]
    mask_slice = vol.mask_3d[:, :, z]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    titles = ["DCE ch0 pré", "DCE ch1 post précoce", "DCE ch2 tardif", "Overlay ch1 + masque"]
    for c in range(3):
        axes[c].imshow(img_slice[:, :, c], cmap="gray")
        axes[c].set_title(titles[c])
        axes[c].axis("off")
    axes[3].imshow(overlay_mask_gray(img_slice[:, :, 1], mask_slice))
    axes[3].set_title(titles[3])
    axes[3].axis("off")
    plt.tight_layout()
    fig_path = FIG_DIR / "01_mri_dce_precheck_channels_overlay.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()

    precheck = {
        "dataset": sample_row["dataset"],
        "patient_id": sample_row["patient_id"],
        "volume_id": sample_row["volume_id"],
        "image_path": sample_row["image_path"],
        "mask_path": sample_row["mask_path"],
        "n_phases": vol.n_phases,
        "phase_indices": list(vol.phase_indices),
        "phase_fallback": vol.phase_fallback,
        "spacing_mm": list(vol.spacing_mm),
        "figure": str(fig_path),
    }
    with open(LOG_DIR / "dce_precheck_sample.json", "w", encoding="utf-8") as f:
        json.dump(precheck, f, indent=2, ensure_ascii=False)
    print(json.dumps(precheck, indent=2, ensure_ascii=False))


## Étape 8 — Génération des crops ROI 256


In [ ]:


manifest_rows = []
failures = []
dce_records = []

def save_npz(sample_id: str, split: str, dataset: str, image: np.ndarray, mask: np.ndarray) -> str:
    subdir = NPZ_DIR / dataset / split
    subdir.mkdir(parents=True, exist_ok=True)
    path = subdir / f"{sample_id}.npz"
    np.savez_compressed(path, image=image.astype(np.float32), mask=mask.astype(np.uint8))
    return str(path.relative_to(OUT_ROOT))

if RUN_GENERATION:
    for idx, row in pairs_df.reset_index(drop=True).iterrows():
        try:
            vol = load_volume_record(row)
            img4 = vol.image_4d
            mask3 = vol.mask_3d

            dce_records.append({
                "dataset": row["dataset"],
                "patient_id": row["patient_id"],
                "volume_id": row["volume_id"],
                "n_phases": vol.n_phases,
                "phase_indices": ",".join(map(str, vol.phase_indices)),
                "phase_fallback": vol.phase_fallback,
                "spacing_mm": ",".join(map(str, vol.spacing_mm)),
                "affine_shape_ok": bool(vol.affine_shape_ok),
            })

            slice_nonzero = mask3.reshape(-1, mask3.shape[-1]).sum(axis=0)
            lesion_slices = np.where(slice_nonzero > 0)[0]

            if len(lesion_slices) == 0:
                failures.append({
                    "volume_id": row["volume_id"],
                    "patient_id": row["patient_id"],
                    "dataset": row["dataset"],
                    "reason": "mask_without_lesion_voxels",
                    "image_path": row["image_path"],
                    "mask_path": row["mask_path"],
                })
                continue

            H, W, Z, C = img4.shape

            for z in lesion_slices:
                z = int(z)
                img2d3 = img4[:, :, z, :]
                mask2d = mask3[:, :, z]

                bbox = bbox_from_mask(mask2d)
                if bbox is None:
                    continue
                x0, y0, x1, y1 = bbox
                coords = square_crop_coords(x0, y0, x1, y1, H=H, W=W, margin=ROI_MARGIN)
                img_crop, mask_crop = crop_pad_resize_image_mask(img2d3, mask2d, coords)

                if mask_crop.sum() == 0:
                    failures.append({
                        "volume_id": row["volume_id"],
                        "patient_id": row["patient_id"],
                        "dataset": row["dataset"],
                        "slice_index": z,
                        "reason": "empty_mask_after_resize",
                    })
                    continue

                sample_id = f"{row['dataset']}_{row['patient_id']}_{row['volume_id']}_z{z:04d}"
                sample_id = normalize_text(sample_id)

                rel_npz = save_npz(sample_id, row["split"], row["dataset"], img_crop, mask_crop)

                lesion_ratio_original = lesion_ratio(mask2d)
                lesion_ratio_crop = lesion_ratio(mask_crop)
                bbox_w = x1 - x0
                bbox_h = y1 - y0
                _, _, _, _, crop_size_native, crop_touches_border = coords

                oracle_crop_flag = False if row["split"] == "train" else True

                manifest_rows.append({
                    "sample_id": sample_id,
                    "dataset": row["dataset"],
                    "source": "MAMA-MIA",
                    "split": row["split"],
                    "patient_id": row["patient_id"],
                    "case_id": row["volume_id"],
                    "volume_id": row["volume_id"],
                    "laterality": "",
                    "oracle_crop_flag": bool(oracle_crop_flag),
                    "lesion_ratio_original": lesion_ratio_original,
                    "lesion_ratio_crop": lesion_ratio_crop,
                    "bbox_w": int(bbox_w),
                    "bbox_h": int(bbox_h),
                    "crop_size": int(crop_size_native),
                    "crop_size_native": int(crop_size_native),
                    "crop_touches_border": bool(crop_touches_border),
                    "npz_path": rel_npz,
                    "modality": "DCE-MRI",
                    "dce_phase": "pre,early_post,late_post",
                    "dce_phase_indices": ",".join(map(str, vol.phase_indices)),
                    "dce_phase_fallback": vol.phase_fallback,
                    "slice_index": z,
                    "plane": "axial",
                    "spacing_mm": ",".join(map(str, vol.spacing_mm)),
                })

        except Exception as e:
            failures.append({
                "volume_id": row.get("volume_id", ""),
                "patient_id": row.get("patient_id", ""),
                "dataset": row.get("dataset", ""),
                "reason": type(e).__name__,
                "message": str(e),
                "image_path": row.get("image_path", ""),
                "mask_path": row.get("mask_path", ""),
            })

        if (idx + 1) % 25 == 0:
            print(f"Processed volumes: {idx+1}/{len(pairs_df)} | crops: {len(manifest_rows)} | failures: {len(failures)}")

    manifest_df = pd.DataFrame(manifest_rows)
    failures_df = pd.DataFrame(failures)
    dce_df = pd.DataFrame(dce_records)

    manifest_path = OUT_ROOT / "roi_mri_manifest.csv"
    failures_path = OUT_ROOT / "failures.csv"
    dce_path = LOG_DIR / "dce_phase_report.csv"

    manifest_df.to_csv(manifest_path, index=False)
    failures_df.to_csv(failures_path, index=False)
    dce_df.to_csv(dce_path, index=False)

    print("Saved manifest:", manifest_path, "rows:", len(manifest_df))
    print("Saved failures:", failures_path, "rows:", len(failures_df))
    print("Saved DCE report:", dce_path, "rows:", len(dce_df))
else:
    print("RUN_GENERATION=False : génération non lancée.")
    print("Après pré-check OK, remettre RUN_GENERATION=True dans la configuration.")


## Étape 9 — Audit GO/NO-GO


In [ ]:


def sha256_array(arr: np.ndarray) -> str:
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(arr).tobytes())
    return h.hexdigest()

def validate_npz(npz_path: Path) -> Dict[str, Any]:
    out = {
        "npz_path": str(npz_path),
        "ok": False,
        "reason": "",
        "image_hash": "",
        "mask_hash": "",
        "pair_hash": "",
    }
    try:
        z = np.load(npz_path)
        if "image" not in z or "mask" not in z:
            out["reason"] = "missing_image_or_mask_key"
            return out
        img = z["image"]
        mask = z["mask"]

        checks = []
        checks.append(("image_shape", img.shape == (IMAGE_SIZE, IMAGE_SIZE, 3)))
        checks.append(("mask_shape", mask.shape == (IMAGE_SIZE, IMAGE_SIZE)))
        checks.append(("image_dtype", img.dtype == np.float32))
        checks.append(("mask_dtype", mask.dtype == np.uint8))
        checks.append(("image_finite", bool(np.isfinite(img).all())))
        checks.append(("image_range", float(img.min()) >= -1e-6 and float(img.max()) <= 1 + 1e-6))
        checks.append(("mask_binary", set(np.unique(mask).tolist()).issubset({0, 1})))
        checks.append(("mask_nonempty", int(mask.sum()) > 0))

        failed = [name for name, ok in checks if not ok]
        if failed:
            out["reason"] = ";".join(failed)
            return out

        ih = sha256_array(img)
        mh = sha256_array(mask)
        ph = hashlib.sha256((ih + mh).encode()).hexdigest()

        out.update({
            "ok": True,
            "reason": "ok",
            "image_hash": ih,
            "mask_hash": mh,
            "pair_hash": ph,
        })
        return out
    except Exception as e:
        out["reason"] = f"{type(e).__name__}: {e}"
        return out

if RUN_AUDIT:
    manifest_path = OUT_ROOT / "roi_mri_manifest.csv"
    if not manifest_path.exists():
        print("Manifest non trouvé. Audit complet impossible tant que RUN_GENERATION=True n'a pas été exécuté.")
    else:
        mdf = pd.read_csv(manifest_path)

        counts = (
            mdf.groupby(["dataset", "split"])
            .agg(n_crops=("sample_id", "count"), n_patients=("patient_id", "nunique"), n_volumes=("volume_id", "nunique"))
            .reset_index()
        )
        counts.to_csv(AUDIT_DIR / "counts_by_dataset_split.csv", index=False)

        split_sets = {s: set(mdf.loc[mdf["split"] == s, "patient_id"].astype(str)) for s in mdf["split"].unique()}
        leak_audit = {}
        keys = sorted(split_sets.keys())
        for i, a in enumerate(keys):
            for b in keys[i+1:]:
                leak_audit[f"{a}_vs_{b}"] = len(split_sets[a] & split_sets[b])
        with open(AUDIT_DIR / "patient_leakage_audit.json", "w", encoding="utf-8") as f:
            json.dump(leak_audit, f, indent=2, ensure_ascii=False)

        val_rows = []
        for rel in mdf["npz_path"].astype(str):
            val_rows.append(validate_npz(OUT_ROOT / rel))
        val_df = pd.DataFrame(val_rows)
        val_df.to_csv(AUDIT_DIR / "npz_full_validation.csv", index=False)

        hash_summary = {
            "n_npz": int(len(val_df)),
            "n_ok": int(val_df["ok"].sum()) if len(val_df) else 0,
            "unique_image_hash": int(val_df.loc[val_df["ok"], "image_hash"].nunique()) if len(val_df) else 0,
            "unique_mask_hash": int(val_df.loc[val_df["ok"], "mask_hash"].nunique()) if len(val_df) else 0,
            "unique_pair_hash": int(val_df.loc[val_df["ok"], "pair_hash"].nunique()) if len(val_df) else 0,
        }

        mdf["enrichment"] = mdf["lesion_ratio_crop"] / mdf["lesion_ratio_original"].replace(0, np.nan)
        enrich_stats = mdf[[
            "lesion_ratio_original", "lesion_ratio_crop", "enrichment",
            "bbox_w", "bbox_h", "crop_size_native"
        ]].describe().T.reset_index()
        enrich_stats.to_csv(AUDIT_DIR / "lesion_enrichment_stats.csv", index=False)

        spacing_df = mdf["spacing_mm"].astype(str).str.split(",", expand=True)
        if spacing_df.shape[1] >= 3:
            spacing_df = spacing_df.iloc[:, :3]
            spacing_df.columns = ["spacing_x", "spacing_y", "spacing_z"]
            for c in spacing_df.columns:
                spacing_df[c] = pd.to_numeric(spacing_df[c], errors="coerce")
            spacing_df.describe().T.to_csv(AUDIT_DIR / "spacing_mm_stats.csv")

        dce_path = LOG_DIR / "dce_phase_report.csv"
        dce_summary = {}
        if dce_path.exists():
            dce_df = pd.read_csv(dce_path)
            dce_summary = {
                "n_volumes": int(len(dce_df)),
                "phase_fallback_counts": dce_df["phase_fallback"].value_counts(dropna=False).to_dict() if "phase_fallback" in dce_df else {},
                "n_phase_distribution": dce_df["n_phases"].value_counts(dropna=False).to_dict() if "n_phases" in dce_df else {},
                "affine_shape_ok_rate": float(dce_df["affine_shape_ok"].mean()) if "affine_shape_ok" in dce_df and len(dce_df) else None,
            }

        failures_path = OUT_ROOT / "failures.csv"
        failure_summary = {}
        if failures_path.exists():
            fdf = pd.read_csv(failures_path)
            failure_summary = {
                "n_failures": int(len(fdf)),
                "failure_reason_counts": fdf["reason"].value_counts(dropna=False).to_dict() if "reason" in fdf else {},
            }

        blocking_issues = []
        if any(v > 0 for v in leak_audit.values()):
            blocking_issues.append(f"patient_leakage_detected: {leak_audit}")
        if hash_summary["n_ok"] != hash_summary["n_npz"]:
            blocking_issues.append("invalid_npz_detected")
        if hash_summary["unique_pair_hash"] != hash_summary["n_ok"]:
            blocking_issues.append("duplicate_image_mask_pairs_detected")
        if len(mdf) == 0:
            blocking_issues.append("empty_manifest")
        if "external" not in set(mdf["split"].astype(str)):
            blocking_issues.append("external_split_missing")

        verdict = "GO" if len(blocking_issues) == 0 else "NO-GO"

        audit = {
            "dataset": "ROI_MRI_Crops_256_v1",
            "verdict": verdict,
            "blocking_issues": blocking_issues,
            "counts_by_dataset_split": counts.to_dict(orient="records"),
            "patient_leakage": leak_audit,
            "npz_validation": hash_summary,
            "dce_summary": dce_summary,
            "failure_summary": failure_summary,
            "contract": {
                "image": "float32, shape 256x256x3, range [0,1]",
                "mask": "uint8, shape 256x256, binary non-empty",
                "crop": "GT bbox + 40% margin, square, zero-padding, resize 256",
                "normalization": f"per-volume common robust scale P{P_LOW}/P{P_HIGH}, same scale for 3 DCE phases",
                "no_training": True,
            }
        }

        with open(AUDIT_DIR / "AUDIT_ROI_MRI_256_v1.json", "w", encoding="utf-8") as f:
            json.dump(audit, f, indent=2, ensure_ascii=False)

        md_lines = []
        md_lines.append("# Audit — ROI_MRI_Crops_256_v1\n")
        md_lines.append(f"## Verdict\n\n**{verdict}**\n")
        if blocking_issues:
            md_lines.append("### Blocking issues\n")
            for x in blocking_issues:
                md_lines.append(f"- {x}")
            md_lines.append("")
        md_lines.append("## Counts by dataset/split\n")
        md_lines.append(counts.to_markdown(index=False))
        md_lines.append("\n\n## Patient leakage\n")
        md_lines.append("```json\n" + json.dumps(leak_audit, indent=2, ensure_ascii=False) + "\n```")
        md_lines.append("\n## NPZ validation\n")
        md_lines.append("```json\n" + json.dumps(hash_summary, indent=2, ensure_ascii=False) + "\n```")
        md_lines.append("\n## DCE summary\n")
        md_lines.append("```json\n" + json.dumps(dce_summary, indent=2, ensure_ascii=False) + "\n```")
        md_lines.append("\n## Failure summary\n")
        md_lines.append("```json\n" + json.dumps(failure_summary, indent=2, ensure_ascii=False) + "\n```")
        md_lines.append("\n## Dataset card\n")
        md_lines.append("- Task: binary breast lesion segmentation in DCE-MRI ROI/oracle-crop setting.")
        md_lines.append("- Image: 256×256×3 DCE channels [pre, early post, late post].")
        md_lines.append("- Mask: 256×256 binary lesion mask.")
        md_lines.append("- External sealed set: Duke.")
        md_lines.append("- No model training was performed in this phase.")
        md_lines.append("")
        report_path = OUT_ROOT / "AUDIT_ROI_MRI_256_v1.md"
        with open(report_path, "w", encoding="utf-8") as f:
            f.write("\n".join(md_lines))

        print("Audit verdict:", verdict)
        print("Saved:", report_path)
        display(counts)


## Étape 10 — ZIP léger


In [ ]:


def make_zip(zip_path: Path, include_npz: bool = True):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for p in OUT_ROOT.rglob("*"):
            if p.is_dir():
                continue
            rel = p.relative_to(OUT_ROOT)
            if not include_npz and str(rel).startswith("npz/"):
                continue
            zf.write(p, arcname=f"ROI_MRI_Crops_256_v1/{rel}")
    return zip_path

if CREATE_ZIP:
    light_zip = Path("/kaggle/working/ROI_MRI_Crops_256_v1_AUDIT_LIGHT.zip")
    if (OUT_ROOT / "roi_mri_manifest.csv").exists():
        make_zip(light_zip, include_npz=False)
        print("Saved light ZIP:", light_zip)
    else:
        print("Manifest absent. ZIP non créé.")
